# Lab 09-08 · The MOS Capacitor

Companion to chapter [09-08](../../chapters/09-08-mos-capacitor.md). Handout: [README.md](README.md).

The capacitor: p-type silicon with $N_A$ = 10¹⁷ cm⁻³, a 10 nm SiO₂ dielectric and an n⁺ polysilicon gate, at 300 K.

In [1]:
import math
import numpy as np
from scipy.integrate import quad, solve_ivp, trapezoid
from scipy.optimize import brentq, curve_fit
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

Q = 1.602176634e-19
VT = 1.380649e-23 * 300 / Q
NI = 1e10
EPS = 11.7 * 8.8541878128e-14
EOX = 3.9 * 8.8541878128e-14
NA, TOX = 1e17, 10e-7
COX = EOX / TOX
N0 = NI**2 / NA
VFB = 4.05 - (4.05 + 1.12 - VT * math.log(1.8e19 / NA))
PB = VT * math.log(NA / NI)
WDM = math.sqrt(2 * EPS * 2 * PB / (Q * NA))
VT0 = VFB + 2 * PB + Q * NA * WDM / COX
print(f"V_fb = {VFB:.4f} V, 2phi_B = {2*PB:.4f} V, V_t = {VT0:.4f} V")
assert abs(VFB + 0.9858) < 5e-5 and abs(VT0 - 0.3293) < 5e-5

def Qs(phi):
    if abs(phi) < 1e-12:
        return 0.0
    F2 = (math.exp(-phi / VT) + phi / VT - 1) + (N0 / NA) * (math.exp(phi / VT) - phi / VT - 1)
    return -math.copysign(1, phi) * math.sqrt(2 * Q * NA * EPS * VT * F2)

V_fb = -0.9858 V, 2phi_B = 0.8334 V, V_t = 0.3293 V


## Part 1 — Poisson's equation in the silicon

At a chosen surface potential, integrate $\phi'' = -\rho/\varepsilon_s$ from the surface inward, starting with the surface field from the first integral. The potential should decay to zero in the bulk.

*Predict first:* at threshold, is the electron sheet density larger or smaller than $N_A L_D$?

In [2]:
def profile(phis, depth=0.25e-4, n=4000):
    Es = -Qs(phis) / EPS
    rhs = lambda x, y: [y[1], (Q / EPS) * (NA * (1 - math.exp(-y[0] / VT)) + N0 * (math.exp(y[0] / VT) - 1))]
    x = np.linspace(0, depth, n)
    sol = solve_ivp(rhs, [0, depth], [phis, -Es], t_eval=x, rtol=1e-10, atol=1e-14, method="Radau")
    return sol.t, sol.y[0]

x, phi = profile(2 * PB)
n = N0 * np.exp(phi / VT)
xs = x[x < 0.04e-4]
ns_sheet = trapezoid(n[x < 0.04e-4], xs)
LD = math.sqrt(EPS * VT / (Q * NA))
print(f"electron sheet density at threshold: {ns_sheet:.3e} cm^-2;  N_A L_D = {NA*LD:.3e} cm^-2")
print(f"exact |Q_s| at 2 phi_B: {abs(Qs(2*PB)):.4e} C/cm^2 (depletion approximation {Q*NA*WDM:.4e})")
assert abs(abs(Qs(2 * PB)) - 1.663e-7) < 5e-11
plt.semilogy(x * 1e7, n, label="n"); plt.semilogy(x * 1e7, NA * np.exp(-phi / VT), label="p")
plt.xlim(0, 150); plt.xlabel("depth (nm)"); plt.ylabel("density (cm^-3)"); plt.legend(); plt.show()

electron sheet density at threshold: 1.650e+10 cm^-2;  N_A L_D = 1.293e+11 cm^-2
exact |Q_s| at 2 phi_B: 1.6632e-07 C/cm^2 (depletion approximation 1.6632e-07)


## Part 2 — Charges against gate voltage

*Predict first (the chapter's guided exercise):* how far above $2\phi_B$ is $\phi_s$ when the inversion charge equals the depletion charge, and is $V_g$ there larger or smaller than $V_t + |Q_{dep}|/C_{ox}$?

In [3]:
qb = lambda p: math.sqrt(2 * Q * EPS * NA * (p - VT))
qinv = lambda p: abs(Qs(p)) - qb(p)
vg = lambda p: VFB + p - Qs(p) / COX
pe = brentq(lambda p: qinv(p) - qb(p), 2 * PB, 1.5)
p11 = brentq(lambda p: qinv(p) - Q * 1e11, 0.6, 1.2)
print(f"inversion = depletion at phi_s - 2phi_B = {(pe-2*PB)*1e3:.1f} mV, V_g = {vg(pe):.4f} V; sheet model says {VT0 + Q*NA*WDM/COX:.4f} V")
print(f"V_g for 1e11 electrons/cm^2: {vg(p11):.3f} V (2phi_B threshold {VT0:.4f} V)")
p2 = brentq(lambda p: vg(p) - 2.0, 0.5, 1.5)
print(f"phi_s at V_g = 2 V: {p2:.4f} V")
assert abs((pe - 2 * PB) * 1e3 - 121.0) < 0.05 and abs(vg(pe) - 0.9854) < 5e-5
assert abs(vg(p11) - 0.431) < 5e-4 and abs(p2 - 0.9947) < 5e-5

inversion = depletion at phi_s - 2phi_B = 121.0 mV, V_g = 0.9854 V; sheet model says 0.8109 V
V_g for 1e11 electrons/cm^2: 0.431 V (2phi_B threshold 0.3293 V)
phi_s at V_g = 2 V: 0.9947 V


## Part 3 — C–V curves and extraction

The quasi-static curve uses the full $Q_s$. The high-frequency curve removes the electron term below $2\phi_B$ and freezes the depletion edge at $W_{d\max}$ above it.

*Predict first:* which feature of the curve is most sensitive to doping?

In [4]:
phis = np.linspace(-0.25, 1.05, 2000)
dp = 1e-6
vgs = np.array([vg(p) for p in phis])
cs = np.array([-(Qs(p + dp) - Qs(p - dp)) / (2 * dp) for p in phis])
c_lf = 1 / (1 / COX + 1 / cs)
def Qs_noinv(p):
    return -math.copysign(1, p) * math.sqrt(2 * Q * NA * EPS * VT * (math.exp(-p / VT) + p / VT - 1)) if abs(p) > 1e-12 else 0.0
c_hf = np.array([1 / (1 / COX + 1 / (-(Qs_noinv(p + dp) - Qs_noinv(p - dp)) / (2 * dp))) if p < 2 * PB
                 else 1 / (1 / COX + WDM / EPS) for p in phis])
cfb = 1 / (1 / COX + LD / EPS)
print(f"C_FB/C_ox = {cfb/COX:.4f}, C_min/C_ox = {c_hf[-1]/COX:.4f}")
assert abs(cfb / COX - 0.6988) < 5e-5 and abs(c_hf[-1] / COX - 0.2242) < 5e-5
plt.plot(vgs, c_lf / COX, label="quasi-static"); plt.plot(vgs, c_hf / COX, label="high frequency")
plt.xlabel("V_g (V)"); plt.ylabel("C/C_ox"); plt.legend(); plt.show()
# synthetic noisy measurement and extraction
rng = np.random.default_rng(3)
meas = c_hf * (1 + 0.002 * rng.standard_normal(c_hf.size))
cmax, cmin = np.max(meas[:200]), np.mean(meas[-200:])
tox_x = EOX / cmax
def cmin_ratio(N, cox):
    W = math.sqrt(2 * EPS * 2 * VT * math.log(N / NI) / (Q * N)); return 1 / (1 / cox + W / EPS) / cox
N_x = brentq(lambda N: cmin_ratio(N, cmax) - cmin / cmax, 1e14, 1e19)
print(f"extracted: T_ox = {tox_x*1e7:.2f} nm (true 10 nm; noise and the incompletely saturated accumulation make this approximate), N_A = {N_x:.3e}")

C_FB/C_ox = 0.6988, C_min/C_ox = 0.2242
extracted: T_ox = 9.99 nm (true 10 nm; noise and the incompletely saturated accumulation make this approximate), N_A = 9.988e+16


## Part 4 — Interface charge and poly depletion

*Predict first:* which way does a positive interface charge of 10¹¹ cm⁻² shift the curve, and by how much?

In [5]:
shift = -Q * 1e11 / COX
print(f"flat-band shift from 1e11 cm^-2 positive charge: {shift*1e3:.1f} mV")
assert abs(shift * 1e3 + 46.4) < 0.05
plt.plot(vgs, c_hf / COX, label="ideal"); plt.plot(vgs + shift, c_hf / COX, label="with 1e11 cm^-2 positive charge")
plt.xlabel("V_g (V)"); plt.ylabel("C/C_ox"); plt.legend(); plt.show()
# poly depletion on a 2 nm oxide: inversion capacitance against oxide field
tox2 = 2e-7
for Np in (5e19, 1e20, 3e20):
    E = np.linspace(1e6, 6e6, 6)
    Wp = EOX * E / (Q * Np)
    C = 1 / (tox2 / EOX + Wp / EPS)
    print(f"N_poly = {Np:.0e}: C/C_ox at 5 MV/cm = {C[-2] / (EOX/tox2):.3f}")

flat-band shift from 1e11 cm^-2 positive charge: -46.4 mV
N_poly = 5e+19: C/C_ox at 5 MV/cm = 0.736
N_poly = 1e+20: C/C_ox at 5 MV/cm = 0.848
N_poly = 3e+20: C/C_ox at 5 MV/cm = 0.944


## Questions

Answer the discussion questions in the [lab handout](README.md), then try the optional extension (interface traps).